# Referenzquelle NIBRS: die gemeinsame Deliktsprache

Diese Quelle liefert keine Zeilen, sondern **Bedeutung**. Chicago und Detroit
zählen dieselben Taten unter verschiedenen Schlüsseln; ohne eine gemeinsame
Klassifikation ist jeder Vergleich eine Behauptung.

**Pipeline-Stufe:** Referenzdokumentation. Es gibt keine Ingestion — die
Zuordnung liegt als zwei gepflegte Tabellen in `config/` und wird in Silver
angewandt.

| | |
| --- | --- |
| Herausgeber | FBI, gemeinsam mit dem Bureau of Justice Statistics |
| Rolle | nationaler Standard der Kriminalitätserfassung der USA seit Januar 2021 |
| Vorgänger | Summary Reporting System (SRS), zum 1.1.2021 eingestellt |
| Im Projekt | Zielsprache für Chicagos IUCR und Detroits MICR |

**Was dieses Notebook zeigt:** dass die Sache eine Falle enthält, die man
übersieht, wenn man nur auf Spaltennamen schaut — und wie die beiden
Mapping-Tabellen des Projekts damit umgehen. Die Quellenverweise stehen am
Ende in Abschnitt 8.

## 1. Was NIBRS ist

**NIBRS** steht für *National Incident-Based Reporting System*. Es ist seit
Januar 2021 der nationale Standard für die Meldung von Kriminalitätsdaten in
den USA und wird von FBI und BJS gemeinsam betrieben.

Der entscheidende Unterschied zum Vorgänger liegt im Namen: **incident-based**.
Das SRS zählte je Vorfall nur die schwerste Straftat — die sogenannte
Hierarchieregel. NIBRS erfasst mehrere Delikte je Vorfall, dazu Angaben zu
Opfern, Tatverdächtigen, Waffen und Sachschaden.

Für dieses Projekt ist das mehr als eine Fussnote: Ein Datensatz mit
Hierarchieregel zählt weniger Delikte als einer ohne, **bei identischer
Kriminalität**. Wer zwei Quellen vergleicht, die diese Regel unterschiedlich
anwenden, vergleicht Erfassungsverfahren und nicht Taten.

Genau deshalb führt das Projekt in Silver `event_key` und `offense_key`
getrennt: Ein Ereignis kann mehrere Delikte tragen.

## 2. Die Falle: Chicagos `fbi_code` ist **nicht** NIBRS

Chicagos Falldaten führen eine Spalte `fbi_code`. Der Name legt nahe, dass dort
der FBI-Standard steht — also NIBRS. Das stimmt nicht: Es ist der **ältere
UCR-/SRS-Schlüssel**.

Die beiden Codesysteme sehen ähnlich aus, sind aber verschiedene Namensräume.
Die nächste Zelle prüft das an den Daten, statt es zu behaupten.

In [ ]:
from pathlib import Path
import csv
import collections

import duckdb

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'config').exists():
    raise RuntimeError('Projekt-Root mit config/ nicht gefunden')

IUCR_MAPPING = PROJECT_ROOT / 'config' / 'iucr_nibrs_mapping.csv'
KATEGORIE_MAPPING = PROJECT_ROOT / 'config' / 'crime_category_mapping.csv'

iucr_map = list(csv.DictReader(IUCR_MAPPING.open(encoding='utf-8')))
kat_map = list(csv.DictReader(KATEGORIE_MAPPING.open(encoding='utf-8')))

con = duckdb.connect()
CHICAGO = f"read_parquet('{PROJECT_ROOT}/data/bronze/city=chicago/**/*.parquet')"
DETROIT = f"read_json_auto('{PROJECT_ROOT}/data/bronze/detroit/*.ndjson')"

print('IUCR-Mapping   :', len(iucr_map), 'Zeilen')
print('Kategorie-Map  :', len(kat_map), 'Zeilen')
print('Chicago-Bronze :', f"{con.execute(f'SELECT count(*) FROM {CHICAGO}').fetchone()[0]:,}", 'Zeilen')
print('Detroit-Bronze :', f"{con.execute(f'SELECT count(*) FROM {DETROIT}').fetchone()[0]:,}", 'Zeilen')

### 2.1 Zwei Namensräume, keine Überschneidung

In [ ]:
fbi_codes = {z[0] for z in con.execute(
    f'SELECT DISTINCT fbi_code FROM {CHICAGO} WHERE fbi_code IS NOT NULL'
).fetchall()}
ibr_codes = {z['ibr_code'] for z in iucr_map if z['ibr_code'].strip()}

print(f'fbi_code in den Falldaten : {len(fbi_codes):>3} verschiedene')
print('   ', ', '.join(sorted(fbi_codes)))
print()
print(f'ibr_code im Mapping       : {len(ibr_codes):>3} verschiedene')
print('   ', ', '.join(sorted(ibr_codes)))
print()
gemeinsam = fbi_codes & ibr_codes
print('gemeinsame Codes:', sorted(gemeinsam) if gemeinsam else 'KEINE')
print()
print('-> zwei getrennte Codesysteme, nicht zwei Schreibweisen desselben')

### 2.2 Dieselbe Zeile, zwei Schlüssel

Beide Systeme beschreiben oft dasselbe — aber mit anderem Code und anderer
Feinheit. `08B` ist der SRS-Code für einfache Körperverletzung, `13B` der
NIBRS-Code dafür.

In [ ]:
zu_ibr = {z['iucr']: (z['ibr_code'], z['ibr_category']) for z in iucr_map}

print(f"{'IUCR':<6}{'primary_type':<24}{'fbi_code':<10}{'ibr_code':<10}ibr_category")
for iucr, typ, fbi, n in con.execute(f'''
    SELECT iucr, any_value(primary_type), any_value(fbi_code), count(*)
    FROM {CHICAGO} GROUP BY iucr ORDER BY count(*) DESC LIMIT 8
''').fetchall():
    code, kategorie = zu_ibr.get(iucr, ('-', '-'))
    print(f'{iucr:<6}{typ[:22]:<24}{fbi:<10}{code:<10}{kategorie[:30]}')

**Warum das zählt.** Die vergleichbare Kategorie des Projekts wird aus
`fbi_code` gebildet, also aus **SRS**, nicht aus NIBRS — siehe Abschnitt 3.2.
Das NIBRS-Mapping existiert daneben und ist feiner (51 gegen 26 Codes).

Das ist kein Fehler, aber es gehört gewusst: Wer sagt „wir harmonisieren über
NIBRS", beschreibt derzeit die feinere Tabelle, nicht den Weg, den die
Kategorien tatsächlich nehmen.

## 3. Die Referenz- und Mapping-Tabellen des Projekts

Sie tun Verschiedenes und werden leicht verwechselt.

| Datei | Von | Nach | Ebene |
| --- | --- | --- | --- |
| `config/nibrs_offense_codes_2025.csv` | — | die 81 amtlichen NIBRS-Codes | Zielliste, kein Mapping |
| `config/iucr_nibrs_mapping.csv` | Chicagos `iucr` | NIBRS `ibr_code` | fein, 54 Codes |
| `config/micr_nibrs_mapping.csv` | Detroits `state_offense_code` | NIBRS `nibrs_code` | fein, 63 Codes |
| `config/crime_category_mapping.csv` | Chicagos `fbi_code`, Detroits `offense_category` | gemeinsame `category` | grob, 28 Kategorien |

Die erste Datei ist der Grund, dass die anderen überhaupt prüfbar sind: Ohne
amtliche Zielliste kann niemand sagen, ob ein Mapping auf einen Code zeigt, den
es gibt. Sie stammt aus Anhang A der *NIBRS Technical Specification 2025* und
liegt als PDF unter `docs/`.

### 3.1 `iucr_nibrs_mapping.csv` — Herkunft nachgewiesen

Die Tabelle hat zehn Spalten. **Fünf davon stammen unverändert aus dem
offiziellen Datensatz des Chicago Data Portal**, die anderen fünf sind unsere
Ergänzung. Die nächste Zelle prüft das gegen die Quelle — sie braucht Netz.

In [ ]:
import requests

BASIS_SPALTEN = {'iucr', 'primary_description', 'secondary_description',
                 'index_code', 'active'}
PORTAL = 'https://data.cityofchicago.org/resource/c7ck-438e.json'

try:
    sitzung = requests.Session()
    sitzung.headers['User-Agent'] = 'crime-data-gruppe3/0.1'
    probe = sitzung.get(PORTAL, params={'$limit': 1}, timeout=60)
    probe.raise_for_status()
    anzahl = sitzung.get(PORTAL, params={'$select': 'count(*)'}, timeout=60).json()
    portal_spalten = set(probe.json()[0])
    portal_zeilen = int(anzahl[0]['count'])

    print('Quelle  :', PORTAL)
    print('Zeilen  :', f'im Portal {portal_zeilen}, bei uns {len(iucr_map)}',
          '-> gleich' if portal_zeilen == len(iucr_map) else '-> ABWEICHUNG')
    print('Spalten :', sorted(portal_spalten))
    print('  identisch mit unseren Basisspalten:',
          portal_spalten == BASIS_SPALTEN)
    print()
    print('unsere Ergaenzung:', sorted(set(iucr_map[0]) - portal_spalten))
except requests.RequestException as fehler:
    print('Portal nicht erreichbar:', fehler)
    print('unsere Spalten:', sorted(iucr_map[0]))

### 3.2 `crime_category_mapping.csv` — die vergleichbare Ebene

Hier laufen beide Städte zusammen. Chicago tritt über `fbi_code` ein, Detroit
über `offense_category`.

In [ ]:
je_stadt = collections.defaultdict(list)
for z in kat_map:
    je_stadt[z['city']].append(z)

for stadt, zeilen in sorted(je_stadt.items()):
    felder = {z['source_field'] for z in zeilen}
    print(f'{stadt:<9}{len(zeilen):>3} Zeilen   Eintrittsfeld: {felder}')

kategorien = {z['category'] for z in kat_map}
vergleichbar = {z['category'] for z in kat_map
                if z['comparable'].strip().lower() == 'true'}
in_beiden = {
    k for k in vergleichbar
    if {z['city'] for z in kat_map
        if z['category'] == k and z['comparable'].strip().lower() == 'true'}
    == {'chicago', 'detroit'}
}
print()
print(f'Kategorien insgesamt      : {len(kategorien)}')
print(f'als vergleichbar markiert : {len(vergleichbar)}')
print(f'in beiden Staedten belegt : {len(in_beiden)}')
print()
print('  ', ', '.join(sorted(vergleichbar)))

**Nur eine Kategorie, die in beiden Städten belegt ist, trägt einen Vergleich.**
Deshalb ist diese Zahl — und nicht die Zahl aller Kategorien — die Grundlage der
Gold-Schicht.

## 4. Woher der Crosswalk kommt — und warum er neu gebaut wurde

Die Spalte `match` hält fest, wie jede Zeile zu ihrem NIBRS-Code kam.

**Die erste Fassung dieser Tabelle war fehlerhaft.** Sie entstand aus einem Join
über den I-UCR-Code auf eine Codetabelle der Illinois State Police. Diese Tabelle
ist nach **ILCS-Statut** geschlüsselt und vergibt je Statut ein eigenes
vierstelliges Kürzel; Chicagos IUCR-Katalog benutzt dieselben Kürzel für etwas
anderes. Bei 86 von 240 gemeinsamen Codes beschrieb das Statut ein völlig
anderes Delikt — und von dort kam der NIBRS-Code.

In [ ]:
herkunft = collections.Counter(z['match'] or '(leer)' for z in iucr_map)
gesamt = len(iucr_map)

erklaerung = {
    'nibrs_definition': 'Chicagos Beschreibung hat eine klare NIBRS-Entsprechung',
    'nibrs_auffang': 'NIBRS kennt das Delikt nicht, deshalb 90Z',
    'nibrs_ermessen': 'zwei Codes waren vertretbar, Begruendung steht in note',
    '(leer)': 'kein Delikt, bewusst nicht zugeordnet',
}
print(f"{'Herkunft':<20}{'Zeilen':>8}{'Anteil':>9}   Bedeutung")
for schluessel, n in herkunft.most_common():
    print(f'{schluessel:<20}{n:>8}{n / gesamt:>9.1%}   {erklaerung.get(schluessel, "")}')

print()
print('Die Ermessensfaelle, jeder mit Begruendung:')
for z in iucr_map:
    if z['match'] == 'nibrs_ermessen':
        print(f"  {z['iucr']} -> {z['ibr_code']:<4} {z['note'][:66]}")

**Kein Code stammt mehr aus einer Positionsentsprechung.** Die Zuordnung
entsteht aus Chicagos eigener Deliktbeschreibung gegen die amtliche Codeliste,
festgehalten in [`tools/build_iucr_nibrs_mapping.py`](../tools/build_iucr_nibrs_mapping.py).
Das Skript erzeugt die Datei reproduzierbar; ein zweiter Lauf liefert dieselben
Bytes.

**106 der 434 Zeilen** bekamen dabei einen anderen Code. Die groben Fehler der
alten Fassung:

| IUCR | Delikt | alt | neu |
| --- | --- | --- | --- |
| `1140` | DECEPTIVE PRACTICE / EMBEZZLEMENT | `250` Forgery | **`270` Embezzlement** |
| `1265` | CRIMINAL DAMAGE / LIBRARY VANDALISM | `370` Pornography | **`290` Vandalism** |
| `0451` | BATTERY / AGG. P.O. – OTHER FIREARM | `35A` Drogen | **`13A`** |
| `0263` | CRIMINAL SEXUAL ASSAULT / KNIFE | `36A` Incest | **`11A` Rape** |
| `1512` | PROSTITUTION / SOLICITING | `11A` Rape | **`40B`** |

Besonders die letzte Zeile: vier Prostitutionsdelikte standen auf einem
Vergewaltigungscode und hätten jede Auswertung der Sexualdelikte verzerrt.

**Die Lehre ist allgemeiner als der Fall.** Ein Schlüssel, der in zwei Systemen
dieselbe *Form* hat, ist deshalb noch nicht derselbe Schlüssel. Ein Join darüber
scheitert nicht — er liefert ein formgültiges und falsches Ergebnis. Dagegen
hilft keine Schemaprüfung, sondern nur ein Abgleich der **Bedeutung**: hier der
Deliktbeschreibungen, die beide Seiten mitliefern.

## 5. Abdeckung: was zugeordnet ist und was nicht

In [ ]:
ohne_zuordnung = [z for z in iucr_map if not z['ibr_code'].strip()]
print(f'ohne NIBRS-Zuordnung: {len(ohne_zuordnung)} von {len(iucr_map)}')
for z in ohne_zuordnung:
    print(f"  {z['iucr']}  {z['primary_description']} / {z['secondary_description']}"
          f"   (aktiv={z['active']})")

if ohne_zuordnung:
    codes = tuple(z['iucr'] for z in ohne_zuordnung)
    treffer = con.execute(
        f'SELECT count(*) FROM {CHICAGO} WHERE iucr IN {codes}'
    ).fetchone()[0]
    print(f'\n  davon in den Falldaten: {treffer}')

in_daten = {z[0] for z in con.execute(
    f'SELECT DISTINCT iucr FROM {CHICAGO}').fetchall()}
bekannt = {z['iucr'] for z in iucr_map}
print(f'\nverschiedene IUCR-Codes in den Falldaten: {len(in_daten)}')
print('nicht in der Mapping-Tabelle:', sorted(in_daten - bekannt) or 'keiner')

**Der Unterschied zwischen zwei Aussagen** ist hier wichtig:

- „Jede Zeile der Tabelle trägt einen NIBRS-Code" — **falsch**, zwei sind offen.
  Beide sind aber bewusst offen: `1481` ist der Entzug einer Waffentragerlaubnis,
  `9901` ein Sammelkennzeichen für häusliche Gewalt. Keines ist ein Delikt.
- „Jeder Code, der in den Falldaten vorkommt, ist zugeordnet" — **richtig**.
- „Jeder Fall trägt ein echtes NIBRS-Delikt" — **falsch**. 2,97 % landen im
  Auffang `90Z`, weil NIBRS das Delikt nicht kennt.

Für H3 zählt die dritte, und sie ist die unbequemste. Wer die zweite für die
dritte hält, behauptet mehr als er gemessen hat.

## 6. Detroits Seite: MICR — und die zweite Codeliste

Michigan meldet über **MICR**, den *Michigan Incident Crime Reporting*-Standard.
Detroits Falldaten führen ihn als `state_offense_code`, daneben eine gröbere
`offense_category`.

**MICR führt zwei Codelisten, nicht eine.** Das Handbuch trennt sie als eigene
Datenfelder:

| Feld | Frage | Beispiel |
| --- | --- | --- |
| *Offense File Class* | Was ist passiert? | `09001` Murder/Non-Negligent Manslaughter |
| *Arrest Charge* (Data Element #46) | Womit wird jemand beschuldigt? | `0901` Family, Gun |

Die Arrest-Charge-Liste ist **unter** den File Classes gegliedert — 833 Arrest
Charges unter 104 File Classes:

```text
09001 – Murder/Non-Negligent Manslaughter     ← File Class, fünfstellig
    Family, Gun                        0901   ← Arrest Charge, vierstellig
    Family, Other Weapon               0902
```

**Detroit veröffentlicht beide in einem vierstelligen Feld.** File Classes sind
fünfstellig und werden gekürzt (`13001` → `1301`), Arrest Charges sind von Haus
aus vierstellig. Dadurch sind **54 Kürzel doppeldeutig**: `0901` ist zugleich die
gekürzte File Class `09001` und der Arrest Charge *Family, Gun*.

Genau daran ist ein erster Extraktionsversuch gescheitert — er lieferte
`0901 → "Willful Killing – Family – Gun"`, also einen Arrest Charge als File
Class. Aufgefallen ist es erst beim Abgleich mit Detroits eigenen
Beschreibungen. Für alle 54 stützt Detroits `offense_description` die
File-Class-Lesart; **keine einzige widerspricht ihr.**

`config/micr_nibrs_mapping.csv` deckt heute **98,10 %** der Detroit-Fälle ab. Die
Spalte `micr_code_type` sagt je Zeile, welche der beiden Listen gemeint ist.

In [ ]:
zahlen = con.execute(f'''
    SELECT count(DISTINCT attributes.offense_category),
           count(DISTINCT attributes.state_offense_code),
           count(DISTINCT trim(attributes.offense_description))
    FROM {DETROIT}
''').fetchone()
print(f'offense_category    {zahlen[0]:>4}   <- crime_category_mapping.csv, grob')
print(f'state_offense_code  {zahlen[1]:>4}   <- MICR, feiner, jetzt ueber micr_nibrs_mapping.csv')
print(f'offense_description {zahlen[2]:>4}')

print('\nMICR-Codes je Kategorie, die groessten:')
for kat, codes, n in con.execute(f'''
    SELECT attributes.offense_category, count(DISTINCT attributes.state_offense_code),
           count(*)
    FROM {DETROIT} GROUP BY 1 ORDER BY 2 DESC LIMIT 5
''').fetchall():
    print(f'  {kat[:30]:<30}{codes:>3} Codes  {n:>7,} Faelle')

### 6.1 Ist die feinere Ebene brauchbar?

Zwei verschiedene Fragen, die man leicht vermischt:

1. **Trägt ein MICR-Code immer dieselbe Kategorie?** Wenn nicht, ist die
   Zuordnung Code → Kategorie mehrdeutig.
2. **Trägt ein MICR-Code immer dieselbe Beschreibung?** Wenn nicht, ist die
   Erfassung innerhalb einer Kategorie uneinheitlich.

Die Zelle prüft beides getrennt.

In [ ]:
mehrdeutig = con.execute(f'''
    SELECT attributes.state_offense_code AS code,
           count(DISTINCT attributes.offense_category) AS kategorien,
           count(*) AS zeilen
    FROM {DETROIT} WHERE attributes.state_offense_code IS NOT NULL
    GROUP BY 1 HAVING count(DISTINCT attributes.offense_category) > 1
    ORDER BY zeilen DESC
''').fetchall()
print(f'MICR-Codes mit mehr als einer Kategorie: {len(mehrdeutig)} von {zahlen[1]}')
for code, kategorien, zeilen in mehrdeutig:
    print(f'  {code}: {kategorien} Kategorien, {zeilen:,} Zeilen')

print('\nZweite Frage - derselbe Code, verschiedene Beschreibungen (MICR 2301):')
for kat, beschr, n in con.execute(f'''
    SELECT attributes.offense_category, trim(attributes.offense_description), count(*)
    FROM {DETROIT} WHERE attributes.state_offense_code = '2301'
    GROUP BY 1,2 ORDER BY 3 DESC
''').fetchall():
    print(f'  {kat:<10}{beschr[:44]:<46}{n:>6,}')

**Die feinere Ebene ist fast sauber.** Nur zwei von 97 Codes überspannen
Kategorien, und bei den Beschreibungen sind die Abweichungen Einzelfälle. MICR wäre also nutzbar — die
Entscheidung für die gröbere Ebene ist eine Abwägung und keine Notwendigkeit.

Was dagegen spricht, MICR jetzt zu nutzen: Für Chicago müsste dann ebenfalls die
feinere Ebene gelten, also IUCR statt `fbi_code`, und damit stünde die
NIBRS-Tabelle mit ihren zwei Fünfteln Handarbeit im Weg zur Vergleichszahl statt
daneben.

### 6.2 Eine Bereinigungsaufgabe, die hier sichtbar wird

In [ ]:
roh, getrimmt, gepolstert, detroit_zeilen = con.execute(f'''
    SELECT count(DISTINCT attributes.offense_description),
           count(DISTINCT trim(attributes.offense_description)),
           count(*) FILTER (attributes.offense_description
                            <> trim(attributes.offense_description)),
           count(*)
    FROM {DETROIT}
''').fetchone()

print(f'Zeilen mit Randleerzeichen: {gepolstert:,} von {detroit_zeilen:,} '
      f'= {gepolstert / detroit_zeilen:.1%}')
print()
print(f'verschiedene Beschreibungen, ungetrimmt: {roh}')
print(f'verschiedene Beschreibungen, getrimmt  : {getrimmt}')
print(f'-> {roh - getrimmt} Scheinwerte allein durch Leerzeichen')
print()
print('Bronze laesst den Text wie geliefert; das Trimmen gehoert nach Silver.')

**Fast die Hälfte der Beschreibungen ist ein Artefakt.** Ungetrimmt zählt die
Quelle 151 verschiedene Deliktbeschreibungen, getrimmt nur 81 — **70 Werte
entstehen allein durch Leerzeichen**.

Wer ohne `trim()` gruppiert, bekommt dieselbe Beschreibung zweimal und hält den
Unterschied womöglich für eine Erfassungsvariante. Das ist genau die Sorte
Fehler, die in einer Auswertung nicht auffällt, weil das Ergebnis plausibel
aussieht.

Bronze lässt den Text wie geliefert — das ist richtig, denn die Auffüllung
gehört zur Quelle. Das Trimmen gehört nach Silver, zusammen mit dem Parsen der
Zeitstempel und dem Auflösen der Fehlwerte.

## 7. Was das für H3 bedeutet

**H3 lautet:** Die Deliktschlüssel beider Städte lassen sich vollständig auf
NIBRS abbilden.

Seit `silver/crime_offense` beide Crosswalks auf die Falldaten anwendet, ist das
gemessen — über alle 347'967 Delikte des Jahrgangs 2024:

| | Chicago | Detroit | zusammen |
| --- | ---: | ---: | ---: |
| Delikte | 259'272 | 88'695 | 347'967 |
| Quellcode im Crosswalk bekannt | **100 %** | **98,1 %** | 99,5 % |
| davon Auffang `90Z` | 7'695 | 3'098 | 10'793 |
| **mit einem echten NIBRS-Delikt** | **97,0 %** | **94,6 %** | **96,4 %** |

**H3 ist teilweise falsifiziert** — nicht weil etwas fehlt, sondern weil
„vollständig" zweierlei heissen kann: Abdeckung ja, Abbildung nein. 10'793 Fälle
tragen ein Delikt, für das NIBRS keinen Code führt, und landen auf `90Z All
Other Offenses` — gültig, für einen Deliktvergleich aber ohne Aussage.

**Die 1'684 gar nicht zugeordneten Fälle** sind fast alle Detroit, und es sind
drei Codes, bei denen Detroits eigene Beschreibung dem Handbuch widerspricht:

| Code | Fälle | Detroit sagt | Handbuch sagt |
| --- | ---: | --- | --- |
| `3072` | 1'477 | RETAIL FRAUD – **THEFT** | Retail Fraud **Misrepresentation** 2nd Degree |
| `5421` | 102 | ACCIDENT, HIT & RUN | Operating **ORV** Under the Influence |
| `5422` | 102 | ACCIDENT, HIT & RUN | Permitted Person Under the Influence |

Bei `3072` steht 26A *False Pretenses* gegen 23C *Shoplifting* — ein erheblicher
Unterschied. Hier zu raten wäre genau der Fehler, der den Chicago-Crosswalk 106
falsche Zeilen gekostet hat. Die Lücke bleibt benannt, bis Detroit sie auflöst.

**Und die Harmonisierung läuft jetzt tatsächlich über NIBRS**, nicht mehr über
`fbi_code` (SRS) und `offense_category`. Die Zutaten aus Abschnitt 3 sind
angewandt, nicht nur vorhanden.

## 8. Quellenverweise

**Standard und Herausgeber**

| Was | Adresse |
| --- | --- |
| NIBRS beim Bureau of Justice Statistics | <https://bjs.ojp.gov/national-incident-based-reporting-system-nibrs> |
| NIBRS beim FBI | <https://www.fbi.gov/how-we-can-help-you/more-fbi-services-and-information/ucr/nibrs> |
| *A Guide to Understanding NIBRS* (FBI, PDF) | <https://www.fbi.gov/file-repository/ucr/a-guide-to-understanding-nibrs.pdf> |
| CRS-Bericht R46668, mit Gegenüberstellung der SRS- und NIBRS-Deliktlisten | <https://www.congress.gov/crs-product/R46668> |

**Illinois und Chicago**

| Was | Adresse |
| --- | --- |
| IUCR-Codes, Datensatz des Chicago Data Portal (`c7ck-438e`) | <https://data.cityofchicago.org/Public-Safety/Chicago-Police-Department-Illinois-Uniform-Crime-R/c7ck-438e> |
| Crime Reporting der Illinois State Police | <https://isp.illinois.gov/CrimeReporting/NIBRS> |
| Illinois-UCR-Portal | <https://ilucr.nibrs.com/> |

**Michigan und Detroit**

| Was | Adresse |
| --- | --- |
| MICR beim Michigan State Police | <https://www.michigan.gov/msp/divisions/cjic/micr> |

**Im Repository — die Belege**

| Was | Wo |
| --- | --- |
| Amtliche NIBRS-Codeliste, 81 Codes | `config/nibrs_offense_codes_2025.csv` |
| Quelle dazu: *NIBRS Technical Specification 2025*, Anhang A | `docs/nibrs_technical_specification_2025.pdf` |
| IUCR nach NIBRS | `config/iucr_nibrs_mapping.csv`, erzeugt von [`tools/build_iucr_nibrs_mapping.py`](../tools/build_iucr_nibrs_mapping.py) |
| MICR nach NIBRS | `config/micr_nibrs_mapping.csv` |
| Quellen dazu | `docs/micr_manual_1110_2024.pdf`, `docs/micr_appendix_f_2009.pdf` |
| Beide Städte nach gemeinsamer Kategorie | `config/crime_category_mapping.csv` |
| Codetabelle der ISP, Fassung 2026 | `docs/isp_nibrs_code_table_2026.xlsx` |
| Abgabetext, Kapitel 3.3 und 8.5 | [`docs/ln1_konzept.md`](../docs/ln1_konzept.md) |

**Zur ISP-Codetabelle im Repository:** Sie liegt als Beleg dabei, **nicht** als
Crosswalk. Sie ist nach ILCS-Statut geschlüsselt und lässt sich nicht über den
I-UCR-Code auf Chicagos Katalog joinen — genau daran ist die erste Fassung des
Crosswalks gescheitert (Abschnitt 4). Warum die Michigan-Dokumente als PDF im
Repository liegen statt nur als Verweis: `michigan.gov` beantwortet
automatisierte Abrufe mit HTTP 403.

**Stand der Abrufe:** Die Angaben zu NIBRS und zum Chicago Data Portal wurden am
10.9.2026 geprüft; der IUCR-Katalog wird von `tools/build_iucr_nibrs_mapping.py`
bei jedem Lauf frisch geladen und gegen die Datei abgeglichen. Die Seite des FBI
antwortete auf automatisierte Abrufe mit HTTP 403; sie ist im Browser erreichbar
und hier als Verweis aufgeführt, nicht als geprüfte Quelle.

## 9. Offen

- **Drei Detroit-Codes**, zusammen 1'681 Fälle: `3072`, `5421`, `5422`. Detroits
  Beschreibung widerspricht dem MICR-Handbuch; das kann nur Detroit auflösen.
- **Vier MICR-Zielcodes sind zurückgezogen** (`90A`, `90E`, `90H`, `90I`). Sie
  bleiben im Crosswalk, weil die MICR-Quellen sie führen, und tragen eine Notiz.
  `7070 Runaway` ist über die File Class auf `90I` angeschlossen und erbt das.
- **Ob Minderjährigkeit allein `64A` Menschenhandel begründet**, ist offen. Die
  *Technical Specification* ist reine Datenspezifikation und enthält keine
  Deliktdefinitionen. Betrifft `1521`, `1526`, `1531`; als Ermessen markiert.
- **Die Rate je Einwohner** — absolute Fallzahlen sind zwischen Städten mit
  unterschiedlicher Einwohnerzahl nicht vergleichbar. Der Nenner steht in
  `silver/tract` bereit; die Verbindung gehört nach Gold.

In [ ]:
con.close()
print('Verbindung geschlossen.')